In [1]:
import sys

if "google.colab" in sys.modules:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    original_data = '/content/drive/My Drive/MS1/original_dataset'
    final_data = '/content/drive/My Drive/MS1/Final_Dataset'

    # Install required packages
    !pip install pymatgen torch_geometric mp_api

else:
    original_data = "original_dataset"
    final_data = "Final_Dataset"

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, NNConv, CGConv, global_max_pool, GCNConv

from pymatgen.core import Structure, PeriodicSite, DummySpecie, Composition, Element
from pymatgen.core.periodic_table import Element as PMGElement
from pymatgen.analysis.local_env import MinimumDistanceNN, CrystalNN, VoronoiNN
# from mp_api.client import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# import json
# import config
# import graphy_gnn
import graphy_cvae

/home/amutua/inverse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Prep

In [3]:
# The whole dataset
comb_df = pd.read_csv(f"{final_data}/combined/combined_data.csv")

# Sample row
sample_row = comb_df.iloc[0]

# Get defective structure
material = sample_row["dataset_material"]
id = sample_row["_id"]

# Get defective structure
defective_file_path = f"{original_data}/{material}/cifs/{id}.cif"
defective_structure = Structure.from_file(defective_file_path)

# Get reference structure
ref_file_path = f"{final_data}/ref_cifs/{material}.cif"
reference_structure = Structure.from_file(ref_file_path)

/home/amutua/inverse/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 32 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


In [4]:
# Create a full defective structure instead of a cloud structure
def struct_to_dict(structure):
    rounded_coords = np.round(structure.frac_coords, 3)
    return {tuple(coord): site for coord, site in zip(rounded_coords, structure.sites)}


def get_defects_structure(defective_struct, reference_struct):
    # mindnn = MinimumDistanceNN()
    # struct to dict
    defective_dict = struct_to_dict(defective_struct)
    reference_dict = struct_to_dict(reference_struct)

    # Get lattice of defective structure
    structure_lattice = defective_struct.lattice

    # List to add all defect sites
    defective_structure_list = []

    # Dictionary to hold properties of each site
    defects_properties = {}

    ref_index = 0

    for ref_coord, ref_site in reference_dict.items():
        # Use the reference coordinates to get the defective site
        ref_index = ref_index + 1

        def_site = defective_dict.get(ref_coord)

        if def_site:  # The site is found in both the reference structure and the defective structure
            # But are the species the same?
            ref_specie = ref_site.specie
            def_specie = def_site.specie
            if ref_specie != def_specie:  # Substitution
                # Add site to defects list
                defective_structure_list.append(def_site)

                # Get atomic number change and defect type
                add_property = {
                    "original_Z":ref_specie.Z,
                    "new_Z": def_specie.Z,

                    "original_en": ref_specie.X,
                    "new_en": def_specie.X,

                    "original_ar": ref_specie.atomic_radius,
                    "new_ar": def_specie.atomic_radius,

                    "original_row": ref_specie.row,
                    "new_row": def_specie.row,

                    "original_group": ref_specie.group,
                    "new_group": def_specie.group,

                    "original_max_os": max(ref_specie.common_oxidation_states),
                    "new_max_os": max(def_specie.common_oxidation_states) if def_specie.common_oxidation_states else 0,

                    "original_ef": ref_specie.electron_affinity,
                    "new_ef": def_specie.electron_affinity,

                    "vacancy_defect": 0.0,
                    "substitution_defect": 1.0,
                    "normal_site":0.0,
                }

                defects_properties[def_site] = add_property
            else: # Normal site
                defective_structure_list.append(ref_site)

                add_property = {
                    "original_Z":ref_specie.Z,
                    "new_Z": ref_specie.Z,

                    "original_en": ref_specie.X,
                    "new_en": ref_specie.X,

                    "original_ar": ref_specie.atomic_radius,
                    "new_ar": ref_specie.atomic_radius,

                    "original_row": ref_specie.row,
                    "new_row": ref_specie.row,

                    "original_group": ref_specie.group,
                    "new_group": ref_specie.group,

                    "original_max_os": max(ref_specie.common_oxidation_states),
                    "new_max_os": max(ref_specie.common_oxidation_states),

                    "original_ef": ref_specie.electron_affinity,
                    "new_ef": ref_specie.electron_affinity,

                    "vacancy_defect": 0.0,
                    "substitution_defect": 0.0,
                    "normal_site":1.0
                }

                defects_properties[ref_site] = add_property

    
        else: # the site from ref_structure is not found in defective structure
            # This means that the site is a vacancy site
            # Add site to defective structure
            vacant_site = PeriodicSite(
                species= DummySpecie(),
                coords= ref_coord,
                coords_are_cartesian= False,
                lattice= structure_lattice
                )

            # Add site to defects list
            defective_structure_list.append(vacant_site)

            ref_specie = ref_site.specie

            # Get atomic number change and defect type
            add_property={
                "original_Z":ref_specie.Z,
                "new_Z": 0,

                "original_en": ref_specie.X,
                "new_en": 0.0,

                "original_ar": ref_specie.atomic_radius,
                "new_ar": 0.0,

                "original_row": ref_specie.row,
                "new_row": 0,

                "original_group": ref_specie.group,
                "new_group": 0,

                "original_max_os": max(ref_specie.common_oxidation_states),
                "new_max_os": 0,

                "original_ef": ref_specie.electron_affinity,
                "new_ef": 0.0,
                
                "vacancy_defect": 1.0,
                "substitution_defect": 0.0,
                "normal_site":0.0

            }
            defects_properties[vacant_site] = add_property

    # create a defects structure
    defective_struct = Structure.from_sites(defective_structure_list)

    # Add properties to defects structure
    for a_site in defective_struct.sites:
        if a_site in defects_properties.keys():
            a_site.properties.update(defects_properties[a_site])
        else:
            pass

    return defective_struct

# Testing
new_defective_structure = get_defects_structure(defective_structure, reference_structure)

In [5]:
def get_cloud(full_defective_structure):
    cloud_list = []

    all_sites = full_defective_structure.sites

    for site in all_sites:
        if site.properties["normal_site"] != 1.0:

            # Add site to list
            cloud_list.append(site)

    cloud_structure = Structure.from_sites(cloud_list)
        
    return cloud_structure

new_cloud = get_cloud(new_defective_structure)

## Create graphs for ML

In [6]:
# NODES

def get_nodes(defective_struct):
    defect_sites = defective_struct.sites

    nodes = []
    for site in defect_sites:
        coords = site.frac_coords.tolist()
        site_features = [
            site.properties["new_Z"]/94,
            site.properties["new_ar"],
            site.properties["new_ef"],
            site.properties["new_en"]/4,
            site.properties["new_group"]/18,
            site.properties["new_max_os"],
            site.properties["new_row"]/9,
            
            site.properties["original_Z"]/94,
            site.properties["original_ar"],
            site.properties["original_ef"],
            site.properties["original_en"]/4,
            site.properties["original_group"]/18,
            site.properties["original_max_os"],
            site.properties["original_row"]/9,

            site.properties["vacancy_defect"],
            site.properties["substitution_defect"],
            site.properties["normal_site"],
        ]
        nodes.append(coords + site_features)

    return nodes

defective_nodes = get_nodes(new_defective_structure)

In [7]:
structure_lattice = reference_structure.lattice

def tensor_to_structure(tensor_structure, structure_lattice):
    all_sites = []

    all_properties = {}
    
    for row in tensor_structure:
        row = row.detach().cpu()
        points = {}
        
        frac_coords = row[:3].clamp(0.0, 1.0).numpy()

        # points["Zd"] = int((row[3] * 94).round().clamp(0, 94).item())
        Zd = int((row[3] * 94).round().clamp(0, 94).item())
        points["Zd"] = Zd
        points["ar_d"] = row[4].item()
        points["ef_d"] = row[5].item()
        points["en_d"] = int((row[6] * 4).round().clamp(0, 4).item())
        points["group_d"] = int((row[7] * 18).round().clamp(0, 18).item())
        points["max_os_d"] = row[8].item()
        points["row_d"] = int((row[9] * 9).round().clamp(0, 9).item())

        points["Zp"] = int((row[10] * 94).round().clamp(0, 94).item())
        points["ar_p"] = row[11].item()
        points["ef_p"] = row[12].item()
        points["en_p"] = int((row[13] * 4).round().clamp(0, 4).item())
        points["group_p"] = int((row[14] * 18).round().clamp(0, 18).item())
        points["max_os_p"] = row[15].item()
        points["row_p"] = int((row[16] * 9).round().clamp(0, 9).item())

        points["vac_defect"] = int(row[17].item())
        points["sub_defect"] = int(row[18].item())
        points["norm_site"] = int(row[19].item())


        the_site = PeriodicSite(
            species= Element.from_Z(Zd).symbol if Zd > 0 else DummySpecie(),
            coords= frac_coords,
            coords_are_cartesian= False,
            lattice= structure_lattice
            )

        all_sites.append(the_site)
        all_properties[the_site] = points

    result_structure = Structure.from_sites(all_sites)

    # for a_site in result_structure.sites:
        # a_site.properties.update(all_properties[a_site])

    return result_structure

attempt = torch.tensor(defective_nodes, dtype=torch.float)
part1 = tensor_to_structure(attempt, structure_lattice)

In [8]:
def get_edges_and_features(reference_structure):
    src, dst = [], []
    features = []
    coords = np.array([s.coords for s in reference_structure.sites])
    new_max_oss = np.array([s.properties["new_max_os"] for s in reference_structure])
    vacancys = np.array([s.properties["vacancy_defect"] for s in reference_structure])
    subs = np.array([s.properties["substitution_defect"] for s in reference_structure])
    norms = np.array([s.properties["normal_site"] for s in reference_structure])

    for i in range(len(reference_structure.sites)):
        for j in range(len(reference_structure.sites)):
            if i == j:
                continue
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < 5.0:
                src.append(i)
                dst.append(j)

                
                 # Electrostatic interaction
                q_i = new_max_oss[i]
                q_j = new_max_oss[j]

                if vacancys[i] == 1:
                    q_i = -q_i

                if vacancys[j] == 1:
                    q_j = -q_j

                charge_product = q_i * q_j
                screened_coulomb = (q_i * q_j) / (dist) if dist > 0 else 0.0

            

                # Defect interaction
                if (vacancys[i] == 1 and subs[j] == 1) or (subs[i] == 1 and vacancys[j] == 1) :
                    vac_sub = 1
                    vac_vac, vac_norm, sub_sub, sub_norm, norm_norm = 0,0,0,0,0

                elif (vacancys[i] == 1 and norms[j] == 1) or (norms[i]==1 and vacancys[j]==1):
                    vac_norm = 1
                    sub_sub, vac_vac, vac_sub, sub_norm, norm_norm = 0,0,0,0,0

                elif (subs[i] == 1 and norms[j] == 1) or (norms[i]==1 and subs[j]==1):
                    sub_norm = 1
                    sub_sub, vac_vac, vac_sub, vac_norm, norm_norm = 0,0,0,0,0

                elif subs[i] == 1 and subs[j] == 1:
                    sub_sub = 1
                    vac_vac, vac_norm, vac_sub, sub_norm, norm_norm = 0,0,0,0,0


                elif vacancys[i] == 1 and vacancys[j]== 1:
                    vac_vac = 1
                    sub_sub, vac_norm, vac_sub, sub_norm, norm_norm = 0,0,0,0,0

                elif norms[i] == 1 and norms[j]== 1:
                    norm_norm = 1
                    vac_vac, vac_norm, vac_sub, sub_norm, sub_sub = 0,0,0,0,0

                else:
                    vac_vac, vac_sub, vac_norm, sub_sub, sub_norm, norm_norm = 0,0,0,0,0,0

                features.append(
                    [
                        dist, charge_product, screened_coulomb, 
                        vac_vac, vac_sub, vac_norm, sub_sub, sub_norm, norm_norm 
                    ]
                )
    ed = [src, dst]
    
    return ed, features

defective_edges, defective_edge_features = get_edges_and_features(new_defective_structure)

In [9]:
data_list = []

the_material = "high_GaSe"
elem = the_material.split("_")[1]

sample_data = comb_df[comb_df["dataset_material"] == the_material].reset_index(drop=True)

all_materials = ["BN", "GaSe", "InSe", "MoS2", "P", "WSe2"]
one_hot = [1 if cat == elem else 0 for cat in all_materials]

ref_sample = Structure.from_file(f"{final_data}/ref_cifs/{the_material}.cif")

for index, row in sample_data.iterrows():
    id = row["_id"]
    bgv = row["band_gap_value"]
    defective_structure = Structure.from_file(f"{original_data}/{the_material}/cifs/{id}.cif")

    full_defective_structure = get_defects_structure(defective_structure, ref_sample)

    the_nodes = get_nodes(full_defective_structure)
    the_edges, the_features = get_edges_and_features(full_defective_structure)

    # cloud_struct = get_cloud(full_defective_structure)

    condition = one_hot + [bgv]

    # global_features = graphy_gnn.get_globals(ref_sample, defective_structure, cloud_struct)

    # band_gap = torch.tensor([bgv], dtype=torch.float)
    
    data = Data(
        x=torch.tensor(the_nodes, dtype=torch.float),
        edge_index=torch.tensor(the_edges, dtype=torch.long),
        edge_attr=torch.tensor(the_features, dtype=torch.float),
        u=torch.tensor(condition, dtype=torch.float).unsqueeze(0),
        # u=torch.tensor(global_features, dtype=torch.float).unsqueeze(0),
        # y=torch.tensor(band_gap, dtype=torch.float).unsqueeze(0)
        )
    
    data_list.append(data)

In [10]:
sample_train, sample_val = train_test_split(data_list, test_size=0.25, random_state=42)

In [11]:
sample_train_loader = DataLoader(sample_train, batch_size=1, shuffle=True) 
sample_val_loader = DataLoader(sample_val, batch_size=1, shuffle=False)

for batch in sample_train_loader:
    print(batch)
    break

DataBatch(x=[144, 20], edge_index=[2, 2232], edge_attr=[2232, 9], u=[1, 7], batch=[144], ptr=[2])


In [12]:
NODE_DIM = sample_train[0].x.shape[1]
EDGE_DIM = sample_train[0].edge_attr.shape[1]
U_DIM = sample_train[0].u.shape[1]

HIDDEN_DIM = 128
LATENT_DIM = 32

# MAX_POINTS = int(MAX_POINTS)

### Old Data Prep

In [21]:
sample_dataset = comb_df[comb_df["dataset_material"]!= "low_MoS2"].reset_index(drop=True)
sample_dataset = sample_dataset[sample_dataset["dataset_material"]!= "low_WSe2"].reset_index(drop=True)

In [22]:
sample_train, sample_val = train_test_split(sample_dataset, test_size=0.3, random_state=42, stratify= sample_dataset['strata'])
sample_train.reset_index(drop=True, inplace=True)
sample_val.reset_index(drop=True, inplace=True)


In [75]:
def fast_cvae_data(dataset):
    data_list = []
    unique_materials = dataset["dataset_material"].unique()
    # run = 0
    # whole_dataset = len(dataset)

    for material in unique_materials:
        subset = dataset[dataset["dataset_material"] == material].reset_index(drop=True)
        reference_structure = Structure.from_file(f"{final_data}/ref_cifs/{material}.cif")

        # Get nodes and edges of pristine strructure
        pristine_nodes_edges = graphy_cvae.func_1(reference_structure)

        for index, row in subset.iterrows():
            id = row["_id"]
            bgv = row["band_gap_value"]
            defective_structure = Structure.from_file(f"{original_data}/{material}/cifs/{id}.cif")

            defects_only_structure = graphy_cvae.get_defects_structure(defective_structure, reference_structure)

            # Defect cloud as tensor
            tensor_cloud = graphy_cvae.cloud_to_tensor(defects_only_structure, MAX_POINTS)

            # band_gap
            band_gap = torch.tensor([bgv], dtype=torch.float32)

            data = Data(
                node_features = pristine_nodes_edges["node_features"],
                edge_index = pristine_nodes_edges["edge_index"],
                defect_cloud = tensor_cloud,
                band_gap = band_gap
            )

            data_list.append(data)

    return data_list

train_dataset = fast_cvae_data(sample_train)
val_dataset = fast_cvae_data(sample_val)
# for_example = fast_cvae_data(sample_dataset.sample(n=50))

/home/amutua/.local/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 48 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/amutua/.local/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 47 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/amutua/.local/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 45 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/amutua/.local/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 46 fractional coordinate

In [76]:
def collate_fn(batch):
    batch_node_features = []
    batch_edge_index = []
    batch_defect_cloud = []
    batch_band_gap = []

    node_offset = 0
    for item in batch:
        batch_node_features.append(item["node_features"])
        batch_defect_cloud.append(item["defect_cloud"])
        batch_band_gap.append(item["band_gap"])

        edge_index = item["edge_index"] + node_offset
        batch_edge_index.append(edge_index)

        node_offset += item["node_features"].shape[0]

    return {
        "node_features": torch.cat(batch_node_features, dim=0),
        "edge_index": torch.cat(batch_edge_index, dim=1),
        "defect_cloud": torch.stack(batch_defect_cloud, dim=0),
        "band_gap": torch.stack(batch_band_gap, dim=0)
    }

In [91]:
test_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

for batch in test_loader:
    print(batch)
    break

DataBatch(edge_index=[2, 3030], node_features=[192, 5], defect_cloud=[24, 7], band_gap=[1], batch=[192], ptr=[2])


In [78]:
NODE_DIM = train_dataset[0].node_features.shape[1]
POINT_DIM = train_dataset[0].defect_cloud.shape[1]
HIDDEN_DIM = 128
LATENT_DIM = 32
MAX_POINTS = int(MAX_POINTS)

## The model architecture

### Model 1

In [88]:
#  ENCODER
class PristineGNNEncoder(nn.Module):
    def __init__(self, node_dim = NODE_DIM, hidden_dim = HIDDEN_DIM, latent_dim = LATENT_DIM, n_max = MAX_POINTS, point_dim = POINT_DIM):
        super().__init__()

        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        cloud_flat = n_max * point_dim
        
        self.fc_combine = nn.Linear(hidden_dim * 2 + cloud_flat, hidden_dim)

        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    # def forward(self, data):
    def forward(self,
            node_features: torch.Tensor,  # (total_nodes, node_dim)
            edge_index:    torch.Tensor,  # (2, total_edges)
            defect_cloud:  torch.Tensor,  # (B * n_max, point_dim)
            batch:         torch.Tensor   # (total_nodes,) – node→graph mapping
            ) -> tuple[torch.Tensor, torch.Tensor]:
        '''node_features = data.node_features
        edge_index = data.edge_index
        defect_cloud = data.defectt_cloud
        batch = data.batch '''

        h = F.relu(self.conv1(node_features, edge_index))
        h = F.relu(self.conv2(h, edge_index))

        h_mean = global_mean_pool(h, batch)
        h_max  = global_max_pool(h, batch)
        h_graph = torch.cat([h_mean, h_max], dim=-1)

        B = h_graph.shape[0]
        cloud_flat = defect_cloud.view(B, -1)  # (B, n_max*point_dim)

        combined = torch.cat([h_graph, cloud_flat], dim=-1)
        hidden   = F.relu(self.fc_combine(combined))

        mu      = self.fc_mu(hidden)       # (B, latent_dim)
        log_var = self.fc_log_var(hidden)  # (B, latent_dim)
        return mu, log_var

In [21]:
# Reparametize to decode
def reparameterize(mu, log_var):
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)
    return mu + eps * std

latent_vector = reparameterize(mu, log_var)
print("Latent vector shape:", latent_vector.shape)  # Should be (B, latent

Latent vector shape: torch.Size([1, 32])


In [ ]:
# DECODER
class DefectCloudDecoder1(nn.Module):
    def __init__(self,hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM, n_max=MAX_POINTS, point_dim=POINT_DIM):
        super().__init__()
        self.n_max     = n_max
        self.point_dim = point_dim

        # +1 for band gap conditioning
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, n_max * point_dim),
        )

    def forward(self, z, band_gap):

        if band_gap.dim() == 1:
            band_gap = band_gap.unsqueeze(-1)

        inp = torch.cat([z, band_gap], dim=-1)
        out = self.net(inp)
        out = out.view(-1, self.n_max, self.point_dim)

        # Apply activations per channel
        coords  = torch.sigmoid(out[..., :3])
        z_vals  = torch.sigmoid(out[..., 3:5])
        logits  = out[..., 5:]

        return torch.cat([coords, z_vals, logits], dim=-1)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ACTUAL MODEL(ENCODER + DECODER)
class DefectCVAE(nn.Module):
    def __init__(self, node_dim = NODE_DIM, hidden_dim = HIDDEN_DIM, latent_dim = LATENT_DIM, n_max = MAX_POINTS, point_dim = POINT_DIM):

        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = PristineGNNEncoder(node_dim, hidden_dim, latent_dim, n_max, point_dim)
        self.decoder = DefectCloudDecoder(hidden_dim,latent_dim, n_max, point_dim)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, data):
        mu, log_var = self.encoder(data.node_features, data.edge_index, data.defect_cloud, data.batch)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z, data.band_gap)
        return recon, mu, log_var

    @torch.no_grad()
    def generate(self, target_band_gap, n_samples, device = "cpu"):
        self.eval()
        z = torch.randn(n_samples, self.latent_dim, device=device)
        bg = torch.full((n_samples,), target_band_gap,
                        dtype=torch.float32, device=device)
        cloud_tensors = self.decoder(z, bg)

        results = []
        for i in range(n_samples):
            points = graphy_cvae.tensor_to_cloud(cloud_tensors[i], threshold=0.1)
            results.append(points)
        return results

### Model 2

In [ ]:
#  ENCODER
class PristineGNNEncoder(nn.Module):
    def __init__(self, node_dim = NODE_DIM, hidden_dim = HIDDEN_DIM, latent_dim = LATENT_DIM, n_max = MAX_POINTS, point_dim = POINT_DIM):
        super().__init__()

        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        cloud_flat = n_max * point_dim
        
        self.fc_combine = nn.Linear(hidden_dim * 2 + cloud_flat, hidden_dim)

        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    # def forward(self, data):
    def forward(self,
            node_features: torch.Tensor,  # (total_nodes, node_dim)
            edge_index:    torch.Tensor,  # (2, total_edges)
            defect_cloud:  torch.Tensor,  # (B * n_max, point_dim)
            batch:         torch.Tensor   # (total_nodes,) – node→graph mapping
            ) -> tuple[torch.Tensor, torch.Tensor]:
        '''node_features = data.node_features
        edge_index = data.edge_index
        defect_cloud = data.defectt_cloud
        batch = data.batch '''

        h = F.relu(self.conv1(node_features, edge_index))
        h = F.relu(self.conv2(h, edge_index))

        h_mean = global_mean_pool(h, batch)
        h_max  = global_max_pool(h, batch)
        h_graph = torch.cat([h_mean, h_max], dim=-1)

        B = h_graph.shape[0]
        cloud_flat = defect_cloud.view(B, -1)  # (B, n_max*point_dim)

        combined = torch.cat([h_graph, cloud_flat], dim=-1)
        hidden   = F.relu(self.fc_combine(combined))

        mu      = self.fc_mu(hidden)       # (B, latent_dim)
        log_var = self.fc_log_var(hidden)  # (B, latent_dim)
        return mu, log_var

In [ ]:
# Reparametize to decode
def reparameterize(mu, log_var):
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)
    return mu + eps * std

latent_vector = reparameterize(mu, log_var)
print("Latent vector shape:", latent_vector.shape)  # Should be (B, latent

In [102]:
ALL_ELEMENTS = [el.symbol for el in Element]   # length 118
N_SPECIES    = len(ALL_ELEMENTS)               # 118

def build_species_mask(allowed_symbols):
    """
    Returns a boolean mask of shape (N_SPECIES,).
    True  → species is allowed (logit kept).
    False → species is forbidden (logit set to -inf).
    """
    allowed_set = set(allowed_symbols)
    mask = torch.tensor(
        [el in allowed_set for el in ALL_ELEMENTS],
        dtype=torch.bool
    )
    return mask

In [108]:
ALLOWED_ELEMENTS = ["Mo", "S", "W", "Se", "B", "N", "Ga", "In", "P", "V", "O", "C"]
ALLOWED_MASK = build_species_mask(ALLOWED_ELEMENTS)

N_DEFECT_TYPES = 2 # vacancy and substitution

class DefectCloudDecoder(nn.Module):
    def __init__(
            self, 
            hidden_dim = HIDDEN_DIM, 
            latent_dim = LATENT_DIM, 
            n_max = MAX_POINTS, 
            allowed_species = ALLOWED_ELEMENTS, 
            allowed_mask = ALLOWED_MASK
        ):
        super().__init__()

        self.n_max = n_max
        self.n_defect_types = N_DEFECT_TYPES          # 2: vacancy, substitution
        self.n_species = N_SPECIES               # 118
        self.species_mask = allowed_mask  # boolean mask for allowed species
        self.defect_mask = torch.ones(self.n_defect_types, dtype=torch.bool)  # all defect types allowed by default

        # point_dim = 3 (coords) + 2 (z-vals) + N_DEFECT_TYPES + N_SPECIES
        self.point_dim = NODE_DIM + self.n_defect_types + self.n_species

        
        out_dim = n_max * self.point_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, out_dim),
        )



    def forward(self, z: torch.Tensor, band_gap: torch.Tensor) -> torch.Tensor:
        if band_gap.dim() == 1:
            band_gap = band_gap.unsqueeze(-1)

        B   = z.shape[0]
        inp = torch.cat([z, band_gap], dim=-1)
        out = self.net(inp).view(B, self.n_max, self.point_dim)

        # Split channels
        coords         = torch.sigmoid(out[..., :3])
        z_vals         = torch.sigmoid(out[..., 3:5])
        defect_logits  = out[..., 5 : 5 + self.n_defect_types]
        species_logits = out[..., 5 + self.n_defect_types :]

        # Apply masks → forbidden logits become -inf before any softmax/sampling
        # defect_logits, species_logits = self._apply_masks(defect_logits, species_logits)

        return torch.cat([coords, z_vals, defect_logits, species_logits], dim=-1)

In [109]:
# Test 1st decoder
the_decoder = DefectCloudDecoder(HIDDEN_DIM, LATENT_DIM, MAX_POINTS, POINT_DIM) # .to(device)
reconstructed_cloud = the_decoder(latent_vector, sample_bg)
print("Reconstructed cloud shape:", reconstructed_cloud.shape)  # Should be (B, n_max, point_dim)

Reconstructed cloud shape: torch.Size([1, 24, 125])


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ACTUAL MODEL(ENCODER + DECODER)
class DefectCVAE(nn.Module):
    def __init__(self, node_dim = NODE_DIM, hidden_dim = HIDDEN_DIM, latent_dim = LATENT_DIM, n_max = MAX_POINTS, point_dim = POINT_DIM):

        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = PristineGNNEncoder(node_dim, hidden_dim, latent_dim, n_max, point_dim)
        self.decoder = DefectCloudDecoder(hidden_dim,latent_dim, n_max, point_dim)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, data):
        mu, log_var = self.encoder(data.node_features, data.edge_index, data.defect_cloud, data.batch)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z, data.band_gap)
        return recon, mu, log_var

    @torch.no_grad()
    def generate(self, target_band_gap, n_samples, device = "cpu"):
        self.eval()
        z = torch.randn(n_samples, self.latent_dim, device=device)
        bg = torch.full((n_samples,), target_band_gap,
                        dtype=torch.float32, device=device)
        cloud_tensors = self.decoder(z, bg)

        results = []
        for i in range(n_samples):
            points = graphy_cvae.tensor_to_cloud(cloud_tensors[i], threshold=0.1)
            results.append(points)
        return results

### Model 3

In [24]:
NODE_DIM

20

In [38]:
# Encoder2
class Encoder2(nn.Module):
    def __init__(self, node_dim, edge_dim, u_dim, hidden_dim, latent_dim):
        super().__init__()

        self.edge_nn = nn.Sequential(
            nn.Linear(edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, node_dim * hidden_dim)
        )

        self.conv0 = NNConv(node_dim, hidden_dim, self.edge_nn, aggr='mean')
        # self.conv1 = CGConv(hidden_dim, edge_dim, aggr='mean') # , batch_norm=True)

        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        self.global_embed = nn.Linear(u_dim, hidden_dim)

        # cloud_flat = n_max * point_dim
        
        self.fc_combine = nn.Linear(hidden_dim * 3, hidden_dim)

        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    def forward(self, data):
        x, edge_index, edge_attr, batch, u = (
            data.x,
            data.edge_index,
            data.edge_attr,
            data.batch,
            data.u,
        )
            
        x = F.relu(self.conv0(x, edge_index, edge_attr))
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x,batch)

        u = self.global_embed(u)

        x = torch.cat([x_max, x_mean], dim=1)
        x = torch.cat([x,u], dim=-1)

        hidden   = F.relu(self.fc_combine(x))
        
        mu = self.fc_mu(hidden) 
        log_var = self.fc_log_var(hidden) 
        
        return mu, log_var
    
# Decoder 2
class Decoder2(nn.Module):
    def __init__(self, hidden_dim, latent_dim, node_dim, u_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(latent_dim + u_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, 144*node_dim),
        )

    def forward(self, z, condition_vector):

        if condition_vector.dim() == 1:
            condition_vector = condition_vector.unsqueeze(-1)

        inp = torch.cat([z, condition_vector], dim=-1)

        return self.net(inp)


# ACTUAL MODEL(ENCODER + DECODER)
class DefectCVAE(nn.Module):
    def __init__(self, node_dim=NODE_DIM, edge_dim=EDGE_DIM, u_dim=U_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = Encoder2(node_dim, edge_dim, u_dim, hidden_dim, latent_dim)
        self.decoder = Decoder2(hidden_dim, latent_dim, node_dim, u_dim)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, data):
        mu, log_var = self.encoder(data)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z, data.u)
        
        return recon, mu, log_var
        # return recon

    @torch.no_grad()
    def generate(self, reference_structure, target_band_gap, device = "cpu"):
        self.eval()

        the_lattice = reference_structure.lattice
        elem = reference_structure.reduced_formula

        all_materials = ["BN", "GaSe", "InSe", "MoS2", "P", "WSe2"]
        one_hot = [1 if cat == elem else 0 for cat in all_materials]

        condition = one_hot + [target_band_gap]


        z = torch.randn(1, 32, device=device)
        bg = torch.tensor(condition*1, dtype=torch.float).unsqueeze(0).to(device=device)
        
        defective_tensors = self.decoder(z, bg).view((144, 20))

        new_structure = tensor_to_structure(defective_tensors, the_lattice)

        return new_structure
    


In [40]:
# Test with first batch from loader
for batch in sample_train_loader:
    the_sample = batch
    break

the_model = DefectCVAE()

recon, mu, log_var = the_model(the_sample)
recon = recon.view((144,NODE_DIM))

### Loss Function

In [43]:
# Loss function for cvae
def cvae_loss(recon_cloud, target_cloud, mu, log_var, beta): # =1e-3):
    # Continuous features: coords + Z values
    mse_loss = F.mse_loss(recon_cloud, target_cloud)    

    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())

    total = mse_loss + beta * kl_loss
    return total, mse_loss, kl_loss

cvae_loss(recon, the_sample.x, mu, log_var, 1e-3)

(tensor(2.7745, grad_fn=<AddBackward0>),
 tensor(2.7738, grad_fn=<MseLossBackward0>),
 tensor(0.7062, grad_fn=<MulBackward0>))

## Training and Testing

In [49]:
# Loaders
train_loader = sample_train_loader # DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = sample_val_loader # DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# Parameters for training model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DefectCVAE().to(device)
EPOCHS = 40
beta = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.001)
losses_dict = {"train_loss": [], "val_loss": [], "kl": [], "mse": []}

# Validate function
def validate(model, loader, beta):
    # Model in evaluation mode
    model.eval()
    loss, mse, kl = 0.0, 0.0, 0.0

    with torch.no_grad():
        for batch in loader:
            # batch = {k: v.to(device) for k, v in batch.items()}
            recon, mu, log_var = model(batch.to(device))
            recon = recon.view((144,20))

            total, mse_loss, kl_loss = cvae_loss(recon, batch.x, mu, log_var, beta)

            loss +=total.item() * batch.num_graphs
            mse += mse_loss.item() * batch.num_graphs
            kl += kl_loss.item() * batch.num_graphs

        n = len(loader.dataset)
        totals = {"the_loss": loss/n, "mse": mse/n, "kl": kl/n}
        return totals

# Training loop
for epoch in range(1, EPOCHS+1):
    # Put model in training mode at the start of each epoch
    model.train()
    t_loss, t_mse, t_kl = 0.0, 0.0, 0.0
    current_beta = beta * min(1.0, epoch / EPOCHS)

    for batch in train_loader:
        optimizer.zero_grad()

        # Output of the model
        recon, mu, log_var = model(batch.to(device))
        recon = recon.view((144, 20))

        
        loss, mse, kl = cvae_loss(recon, batch.x, mu, log_var, current_beta)

        loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        t_loss += loss.item()*batch.num_graphs
        t_mse += mse.item()*batch.num_graphs
        t_kl += kl.item()*batch.num_graphs

    n = len(train_loader)
    t_losses = {"the_loss": t_loss/n, "mse": t_mse/n, "kl": t_kl/n}

    # Validate at the end of each epoch
    val_losses = validate(model, val_loader, current_beta)
    scheduler.step()

    print(f"Epoch {epoch:4d}\n\t train: {t_losses['the_loss']:.4f} | {t_losses['mse']:.4f} | {t_losses['kl']:.4f}\n\t val: {val_losses['the_loss']:.4f} | {val_losses['mse']:.4f} | {val_losses['kl']:.4f}")

/home/amutua/inverse/lib/python3.10/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch    1
	 train: 0.1463 | 0.1460 | 11.6823
	 val: 0.0648 | 0.0645 | 10.6415
Epoch    2
	 train: 0.0631 | 0.0627 | 8.8883
	 val: 0.0627 | 0.0623 | 7.6318
Epoch    3
	 train: 0.0618 | 0.0613 | 6.7140
	 val: 0.0613 | 0.0609 | 5.6634
Epoch    4
	 train: 0.0616 | 0.0611 | 4.9242
	 val: 0.0633 | 0.0629 | 4.1854
Epoch    5
	 train: 0.0601 | 0.0596 | 4.0727
	 val: 0.0617 | 0.0612 | 3.5849
Epoch    6
	 train: 0.0599 | 0.0594 | 3.1887
	 val: 0.0602 | 0.0598 | 2.9406
Epoch    7
	 train: 0.0595 | 0.0590 | 2.6183
	 val: 0.0607 | 0.0603 | 2.5637
Epoch    8
	 train: 0.0593 | 0.0588 | 2.3094
	 val: 0.0599 | 0.0594 | 2.2170
Epoch    9
	 train: 0.0589 | 0.0585 | 1.9557
	 val: 0.0590 | 0.0586 | 1.8469
Epoch   10
	 train: 0.0584 | 0.0580 | 1.7489
	 val: 0.0592 | 0.0587 | 1.7691
Epoch   11
	 train: 0.0591 | 0.0586 | 1.5331
	 val: 0.0586 | 0.0583 | 1.3172
Epoch   12
	 train: 0.0582 | 0.0578 | 1.4268
	 val: 0.0583 | 0.0579 | 1.4516
Epoch   13
	 train: 0.0577 | 0.0572 | 1.3844
	 val: 0.0592 | 0.0587 | 1.27

## Inverse Design

In [50]:
def inverse_design(model, pristine, target_band_gap, device):
    model.eval()
    model = model.to(device)

    # Generate defective structure
    gen_def_structure = model.generate(pristine, target_band_gap)

   
    return gen_def_structure

origin = ref_sample = Structure.from_file(f"{final_data}/ref_cifs/high_GaSe.cif")
first_struct = inverse_design(model, origin, 1.3, "cpu")

In [51]:
print(first_struct)

Full Formula (Rb2 Mn1 V1 Zn14 Cr1 Ga15 Fe2 Co9 Cu15 Ni8 Ge26 As15 Se10 Br16 Kr9)
Reduced Formula: Rb2MnVZn14CrGa15Fe2Co9Cu15Ni8Ge26As15Se10Br16Kr9
abc   :  22.914400  22.914400  20.000000
angles:  90.000000  90.000000 120.000012
pbc   :       True       True       True
Sites (144)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  Ni    0.112623  0.056048  0.377355
  1  Ge    0.108528  0.219112  0.379291
  2  Ni    0.108108  0.38448   0.378505
  3  Ge    0.109691  0.551017  0.37814
  4  Co    0.110627  0.714196  0.378738
  5  Ge    0.107597  0.870086  0.378252
  6  Cu    0.274656  0.055681  0.379102
  7  Cr    0.276782  0.221911  0.380982
  8  Co    0.274569  0.383582  0.38129
  9  As    0.273537  0.548997  0.378453
 10  Ge    0.27584   0.718054  0.379689
 11  Cu    0.276124  0.882788  0.381675
 12  Cu    0.439068  0.057326  0.378207
 13  As    0.441728  0.222276  0.379616
 14  Fe    0.437897  0.38488   0.378116
 15  Ge    0.43846   0.54949   0.387496
